# AXI test 7: the HARD million-token test (Gemini 3.8 Flash, 4,000 steps, strict rules)
Model: **Gemini 3.8 Flash**. One long job: **4,000 requests in a row** about a dense service graph (up to 24 services and 48 edges), about **2.9 million tokens**. At every step the model is shown only the exact current state and one request; the framework (the scope filter, `scope_m7`) decides what to keep, and the held state is compared with the right one at every step. Harder than the earlier tests in six ways (dense graph, services with 4 to 8 dependants to delete, a re-ask when a reply cannot be read, a repair that deletes up to 8 dependants, much stricter reading rules, and a dollar guard). The predictions and the 14 reading rules were written and locked before this run in `docs/MILLION_TOKEN_TEST_HARD.md`. My own odds of a full pass are about 20 percent: the numbers are the product either way.

* **Run 1:** 4,000 steps. **Contrast:** 1,200 more (the first 400 requests again with blind, gate and repair_m6). **Control** (optional): 1,000 steps without the re-ask. **Run 2** (optional): 4,000 more on a second seed.
* **No GPU.** Use the free CPU runtime (Runtime menu -> *Change runtime type* -> CPU). **Keep this tab open while a run is going**; Run 1 takes about 1 to 3 hours of calls.
* **Your key** lives in Colab Secrets (Cell 1). It is never printed or saved in a file. Never paste it in the chat.

**This test costs real money, and this notebook never turns billing on.** You do that, in Google AI Studio, for the project your key belongs to. Then:
1. Open the AI Studio **price page** and read the price of **Gemini 3.8 Flash**: dollars per 1,000,000 *input* tokens and dollars per 1,000,000 *output* tokens (thinking tokens are billed as output). Type these two numbers into Cell 5. The numbers I used for my estimate ($0.75 and $3.75) were pasted from another assistant and are **not verified**: use the page's numbers.
2. Type a **budget in dollars** (`BUDGET_USD`) into Cell 5. I suggest **15**. The notebook adds up what the saved answers cost at your two prices, and **refuses the next call once the total reaches the budget**. Everything finished is saved, and you can raise the budget and carry on. My estimate: about $4 for Run 1 and about $10 for everything including Run 2, if the model thinks about 100 tokens a step. The 20-step check in Cell 6 tells you the real figure before you spend it.
3. A second guard inside the notebook: a **cap of 8,000,000 tokens on each run** (Run 1, each contrast file, the control, Run 2), whatever the price. And, outside it, you can set a **budget alert in Google Cloud Billing** for the same project (the menu names may differ). That one is Google's, and it works even if this notebook stops.
* **Free key:** about 20 calls a day are allowed for this model: enough for the 20-step check, not for the test. Set `PAID = False` and run only Cells 1 to 6.

Run each cell **one at a time, top to bottom**, wait for the green tick before the next one. If it stops for any reason, open this notebook again, run **Cells 2 to 5**, then run the cell you were on again: it continues from the last saved step.

### Cell 1: put your key into Colab Secrets (once; no code to run)
1. In Google AI Studio (`aistudio.google.com`), open **API Keys** in the left menu and copy your key.
2. Here in Colab, tap the **key icon** in the left bar (*Secrets*) -> **Add new secret**.
3. Name: `GEMINI_API_KEY` (exactly). Value: paste the key. Switch **Notebook access** on.
Do not paste the key into a cell or into the chat.

### Cell 2: upload the code. A button 'Choose files' appears: pick **only** `axi-colab-million.zip`

In [ ]:
from google.colab import files
up = files.upload()
cands = [n for n in up if n.endswith(".zip") and "million" in n]
assert len(cands) == 1, f"I need exactly one file named like axi-colab-million.zip. I got: {list(up)}"
name = cands[0]
print("using", name)
!rm -rf /content/axi && mkdir -p /content/axi && unzip -q -o "{name}" -d /content/axi
!ls /content/axi/python

### Cell 3: install the Gemini library (about 20 seconds)

In [ ]:
!pip -q install -U google-genai

### Cell 4: connect Google Drive (a pop-up asks permission: tap Connect / Allow). This is where your progress and every reply are kept

In [ ]:
import sys, os
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_million_hard")

### Cell 5: settings, and connect to the model. Fill in the five lines at the top (PAID, the two prices, the budget); THINKING only if Cell 6 tells you to

In [ ]:
import json, time
from collections import deque
from google.colab import userdata
from axi.experiments.api_llm import QuotaStop, ServiceBusy, BudgetStop, build_gemini_llm
from axi.experiments.million import run_million, print_million, smoke_report, MAIN_POLICY, CONTRAST_POLICIES
from axi.experiments.million_hard import (HARD_MODEL, HARD_SEEDS, HARD_WARM, HARD_STEPS, HARD_CONTRAST_STEPS, HARD_CONTROL_STEPS, HARD_REASKS, HARD_DEGREE, HARD_TOKEN_CAP, HARD_THINKING_LIMIT,
                                          HARD_LOWEST_THINKING, HARD_MINUTES_LIMIT, HARD_FREE_DAILY, verdict_hard, headline, report_hard, compare_control, print_hard_report, print_control, print_verdict_hard)

PAID = None        # True ONLY after you turned billing on for this key in Google AI Studio. False = a free key (only the 20-step check is possible).
PRICE_IN = None    # dollars per 1,000,000 INPUT tokens of Gemini 3.8 Flash, read off the AI Studio price page (type a number, e.g. 0.75 if that is what the page says)
PRICE_OUT = None   # dollars per 1,000,000 OUTPUT tokens (thinking tokens are billed as output), from the same page
BUDGET_USD = None  # the most you allow this test to cost in total, in dollars (I suggest 15). The notebook stops by itself when the saved answers have cost this much.
THINKING = None    # None = the model's own default. Cell 6 tells you if it should be "low" instead; change it only then, before Run 1.

assert PAID in (True, False), "Set PAID in the settings at the top of this cell: True if you turned billing on, False for a free key."
if PAID:
    assert PRICE_IN and PRICE_OUT, "Type PRICE_IN and PRICE_OUT (dollars per million tokens) from the AI Studio price page. The notebook has no built-in price on purpose."
    assert BUDGET_USD, "Type BUDGET_USD (the most you allow this test to cost, in dollars; I suggest 15). Without it the notebook will not spend your money."
try: KEY = userdata.get("GEMINI_API_KEY")
except Exception: raise SystemExit("No key found. Do Cell 1: add a Secret named GEMINI_API_KEY and switch Notebook access on.")
TAG = str(THINKING or "default")
CACHE = "axi_million_hard_cache_%s.jsonl" % TAG
print("reply file:", store.restore(CACHE))
guard = dict(budget_usd=BUDGET_USD, price_in=PRICE_IN, price_out=PRICE_OUT) if PAID else {}
llm, MODEL = build_gemini_llm(KEY, HARD_MODEL, store.path(CACHE), daily_limit=None if PAID else HARD_FREE_DAILY, min_interval=0.3 if PAID else 4.5,
                              on_write=lambda: store.persist(CACHE), thinking=THINKING, **guard)
META = dict(model=MODEL, thinking=THINKING)
spent = llm.spent_usd()
print("paid:", PAID, "| thinking:", THINKING or "model default", "| calls today:", llm.calls_today(), "| token cap per run:", f"{HARD_TOKEN_CAP:,}",
      "| budget: $%.2f" % BUDGET_USD if PAID else "", "| already spent on this model with these settings: " + ("$%.2f" % spent if spent is not None else "-"))
RESET = "The free quota resets at midnight Pacific time: 12:30 pm in India until 1 November, 1:30 pm after. Then open this notebook again, run Cells 2 to 5, and run this cell again."
BUSY = "This is Google's server being busy. Wait at least 30 minutes, then run THIS SAME CELL once. Do NOT run it again and again."
def go(fn):
    try: return fn()
    except QuotaStop as e:
        print("\nSTOPPED FOR NOW: " + str(e))
        if isinstance(e, BudgetStop): pass
        elif isinstance(e, ServiceBusy): print(BUSY)
        else: print(RESET)
        return None

def make_progress(name):
    T0, recent = time.time(), deque(maxlen=100)
    def cb(t, n, rec):
        recent.append(rec)
        if t % 100 == 0 or t == n:
            k = len(recent); el = time.time() - T0; sp = llm.spent_usd()
            print(f"step {t}/{n} | {el/60:.0f} min so far | spent {('$%.2f' % sp) if sp is not None else '-'} | last {k}: exact {100*sum(r['exact'] for r in recent)/k:.0f}%, mean distance "
                  f"{sum(r['distance'] for r in recent)/k:.2f}, {sum(r['seconds'] for r in recent)/k:.2f} s a step, {sum(r['think'] for r in recent)/k:.0f} thinking tokens a step, {sum(r.get('asks', 1) - 1 for r in recent)} re-asks")
    return cb

def save(res, name):
    path = store.path(name + ".json"); json.dump(res, open(path, "w")); store.persist(name + ".json"); return path

def run_file(name, seed, steps, policy, reasks=HARD_REASKS, scripted=0):
    print("progress file:", store.restore(name + ".jsonl"))
    def persist_cb(inner):
        def cb(t, n, rec):
            inner(t, n, rec)
            if t % 25 == 0 or t == n: store.persist(name + ".jsonl")
        return cb
    return go(lambda: run_million(llm, seed, steps, policy, store.path(name + ".jsonl"), meta=META, progress=persist_cb(make_progress(name)), max_tokens=HARD_TOKEN_CAP,
                                  stream="HardStream", reasks=reasks, degree_threshold=HARD_DEGREE, scripted=scripted))

def stop_note(res):
    if res and res["stopped"] == "token_cap": print("STOPPED by the token cap of %s tokens. This is the second guard on the bill; tell me before you raise it." % f"{HARD_TOKEN_CAP:,}"); return None
    return res

### Cell 6: quick check, 20 real calls (about 1 to 2 minutes; a few cents). The program answers the first 200 requests itself, with no call, so that the graph has grown to the size of the real run; then 20 calls go to the model. It prints the tokens a step, the thinking tokens, the time and the cost of the whole run. If it says NOTE, read it before going on

In [ ]:
tag = "axi_million_hard_check_%s" % TAG
smoke = run_file(tag, 99, HARD_WARM + 20, MAIN_POLICY, scripted=HARD_WARM)
if smoke:
    live = smoke["rows"][HARD_WARM:]; sm = smoke["summary"]
    print(f"\ncheck: {len(live)} calls to the model (after {HARD_WARM} steps answered by the program itself): exact {sum(r['exact'] for r in live)}, refused {sum(not r['parsed'] for r in live)}, needed a re-ask {sum(1 for r in live if r.get('asks', 1) > 1)}, "
          f"{sm['tokens']:,} tokens sent ({sm['prompt_tokens']:,} prompt + {sm['output_tokens']:,} reply + {sm['think_tokens']:,} thinking)")
    rep = smoke_report(sm, HARD_STEPS, PRICE_IN, PRICE_OUT, think_limit=HARD_THINKING_LIMIT, lowest=HARD_LOWEST_THINKING, budget_usd=BUDGET_USD, minutes_limit=HARD_MINUTES_LIMIT, token_cap=HARD_TOKEN_CAP)
    if rep["steps"] == 0: print("NOTE:", rep["advice"][0])
    else:
        print(f"{rep['tokens_per_step']:.0f} tokens a step ({rep['think_per_step']:.1f} of them thinking), {rep['seconds_per_step']:.2f} s a step: {HARD_STEPS} steps would be about {rep['projected_tokens'] / 1e6:.2f} million tokens, "
              f"{rep['projected_minutes'] / 60:.1f} hours of calls" + (f", about ${rep['projected_cost']:.2f} for Run 1 at the prices you typed" if rep["projected_cost"] is not None else ""))
        for a in rep["advice"]: print("NOTE:", a)
    for r in live[:6]: print(r["t"], r["kind"], "|", repr(r["text"])[:90])
    if not PAID: print("\nFree key: this is as far as it goes. To run the test, turn billing on in AI Studio, set PAID = True and type the prices and the budget in Cell 5.")

### Cell 7: RUN 1. 4,000 steps, scope_m7. A line is printed every 100 steps; progress and every reply are saved to Drive. If it stops (budget, server busy, your tab closed), run Cells 2 to 5 and this cell again

In [ ]:
assert PAID, "Run 1 needs a paid key: set PAID = True in Cell 5 (after you turned billing on)."
name = "axi_million_hard_run1_seed%d_%s" % (HARD_SEEDS[0], TAG)
res1 = stop_note(run_file(name, HARD_SEEDS[0], HARD_STEPS, MAIN_POLICY))
if res1:
    print_million(res1["summary"], "RUN 1, scope_m7, seed %d, %s" % (HARD_SEEDS[0], MODEL))
    print_verdict_hard(verdict_hard(res1), "RUN 1 (lines 4, 13 and 14 need the next cells)")
    print_hard_report(report_hard(res1), "RUN 1")
    print("totals from the service:", llm.usage(), "| spent so far on this model: $%.2f" % llm.spent_usd())
    files.download(save(res1, name)); files.download(store.path(CACHE))

### Cell 8: CONTRAST. The first 400 requests again with blind, gate and repair_m6 (1,200 calls at most). Also prints the reading rules with lines 4 and 13 and the HEADLINE

In [ ]:
assert res1, "Run 1 has not finished yet. Run Cell 7 again first."
contrast = {}
for p in CONTRAST_POLICIES:
    cname = "axi_million_hard_contrast_%s_seed%d_%s" % (p, HARD_SEEDS[0], TAG)
    r = stop_note(run_file(cname, HARD_SEEDS[0], HARD_CONTRAST_STEPS, p))
    if not r: break
    contrast[p] = r; print_million(r["summary"], "CONTRAST, %s" % p)
if len(contrast) == len(CONTRAST_POLICIES):
    print_verdict_hard(verdict_hard(res1, contrast), "RUN 1 and the contrast")
    print("spent so far on this model: $%.2f" % llm.spent_usd())
    files.download(save(contrast, "axi_million_hard_contrast_seed%d_%s" % (HARD_SEEDS[0], TAG)))

### Cell 9 (optional): CONTROL. scope_m7 WITHOUT the re-ask on the first 1,000 requests: what the re-ask bought. Many of its calls repeat Run 1's and cost nothing

In [ ]:
assert res1, "Finish Cell 7 first."
cname = "axi_million_hard_control_seed%d_%s" % (HARD_SEEDS[0], TAG)
ctl = stop_note(run_file(cname, HARD_SEEDS[0], HARD_CONTROL_STEPS, MAIN_POLICY, reasks=0))
if ctl:
    print_million(ctl["summary"], "CONTROL, scope_m7 without the re-ask")
    print_control(compare_control(res1, ctl))
    print("spent so far on this model: $%.2f" % llm.spent_usd())
    files.download(save(ctl, cname))

### Cell 10 (optional): RUN 2, the second seed, for line 14 (4,000 more steps; about as much money as Run 1). Only if you want the replication

In [ ]:
assert res1 and len(contrast) == len(CONTRAST_POLICIES), "Finish Cells 7 and 8 first."
name2 = "axi_million_hard_run2_seed%d_%s" % (HARD_SEEDS[1], TAG)
res2 = stop_note(run_file(name2, HARD_SEEDS[1], HARD_STEPS, MAIN_POLICY))
if res2:
    print_million(res2["summary"], "RUN 2, scope_m7, seed %d, %s" % (HARD_SEEDS[1], MODEL))
    v = verdict_hard(res1, contrast, res2); print_verdict_hard(v, "RUN 1, the contrast and RUN 2")
    print_hard_report(report_hard(res2), "RUN 2")
    print("totals from the service:", llm.usage(), "| spent so far on this model: $%.2f" % llm.spent_usd())
    files.download(save(res2, name2)); files.download(store.path(CACHE))

### Done
Files downloaded: `axi_million_hard_run1_seed20261031_default.json` (the end of the name says `low` if you changed THINKING), `axi_million_hard_contrast_seed20261031_default.json`, `axi_million_hard_cache_default.jsonl` (and the control and run-2 files if you ran Cells 9 and 10). If a download did not start, the same files are in your Google Drive, in the folder `axi_million_hard`. Upload them to the chat, with a screenshot of the printed tables and the reading-rule lines. Do not paste any keys anywhere. When you are done, you can switch billing off in AI Studio; nothing here does it for you.